# 21. Observability: Logging, Tracing & Monitoring
**Duration:** 35 min | **Topics:** Application Insights, KQL, dashboards

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

| # | Skill | Why It Matters |
|---|-------|----------------|
| 1 | Instrument an LLM app with **structured JSON logging** and `trace_id` on every event | Without trace_id you cannot correlate a request log with its response log across microservices |
| 2 | Add **OpenTelemetry distributed tracing** using W3C `traceparent` headers | A single LLM request can touch 3+ services — tracing shows exactly where latency comes from |
| 3 | Configure **10% sampling** to stay within App Insights 5 GB free tier | At 1000 req/hr, 100% sampling = ~1.4 GB/month — 10% sampling = ~144 MB/month (free) |
| 4 | Write **KQL queries** for latency percentiles, cost per model, and error rate | KQL lets you answer business questions like "which model is costing us the most?" in seconds |
| 5 | Understand **workspace-based App Insights** and its benefits over classic | Workspace-based enables cross-resource KQL joins and long-term retention |
| 6 | Track **LLM-specific metrics**: token usage, estimated cost, hallucination rate | Standard APM metrics do not capture what makes LLM ops unique — you must add them yourself |

---

## 📐 Architecture

```
LLM API request
  │
  ├── Structured log → Azure App Insights customEvents
  │     fields: trace_id, model, latency_ms, tokens, cost_usd
  │
  ├── OpenTelemetry span → App Insights dependencies
  │     W3C traceparent header propagated to downstream services
  │
  └── KQL dashboard → Azure Monitor workbook
        P50/P99 latency | hourly cost | error rate
```

### Minimum Azure Resources
| Resource | SKU | Cost |
|---|---|---|
| Application Insights | Pay-as-you-go | First 5 GB/month **FREE** |
| Log Analytics Workspace | Pay-as-you-go | First 5 GB/month **FREE** |


In [ ]:
import subprocess, sys

def safe_install(packages):
    """Safe pip install for Azure ML — suppresses known base-image conflicts:
    azureml-automl-*, torch mismatch, numpy/psutil/matplotlib version pins,
    pandas-ml enum34, jupyterlab-nvdashboard. None affect notebook code."""
    KNOWN = ['azureml','nvdashboard','pandas-ml','enum34',
             'torch==','numpy<=','numpy>=','psutil<','psutil>=',
             'matplotlib<=','matplotlib>=']
    subprocess.run([sys.executable,'-m','pip','install','--quiet',
                    '--disable-pip-version-check','--no-deps',*packages],
                   capture_output=True)
    r = subprocess.run([sys.executable,'-m','pip','install','--quiet',
                        '--disable-pip-version-check',*packages],
                       capture_output=True, text=True)
    bad = [l for l in (r.stderr or '').splitlines()
           if 'ERROR' in l and not any(k in l for k in KNOWN)]
    print(f'✅ Ready: {", ".join(packages)}') if not bad else print('⚠️', bad)

safe_install(['opencensus-ext-azure', 'azure-monitor-opentelemetry', 'opentelemetry-sdk', 'opentelemetry-api'])


✅ Ready: opencensus-ext-azure, azure-monitor-opentelemetry, opentelemetry-sdk, opentelemetry-api
   (Azure ML env conflicts suppressed — safe to ignore)


## Step 1: Provision Application Insights

In [ ]:
RG          = 'rg-llm-demo'
LOCATION    = 'eastus'
WORKSPACE   = 'law-llm-monitoring'
APP_INSIGHTS= 'appi-llm-demo'

steps = [
    ('1. Create Log Analytics Workspace',
     f'az monitor log-analytics workspace create --resource-group {RG} --workspace-name {WORKSPACE} --location {LOCATION} --sku PerGB2018'),
    ('2. Capture workspace ID',
     f'WORKSPACE_ID=$(az monitor log-analytics workspace show --resource-group {RG} --workspace-name {WORKSPACE} --query id -o tsv)'),
    ('3. Create Application Insights (workspace-based)',
     f'az monitor app-insights component create --app {APP_INSIGHTS} --resource-group {RG} --location {LOCATION} --workspace $WORKSPACE_ID'),
    ('4. Get connection string (set as env var APPLICATIONINSIGHTS_CONNECTION_STRING)',
     f'az monitor app-insights component show --app {APP_INSIGHTS} --resource-group {RG} --query connectionString -o tsv'),
    ('5. Configure sampling to stay under 5 GB free tier',
     f'az monitor app-insights component update --app {APP_INSIGHTS} --resource-group {RG} --sampling-percentage 10'),
]

print('='*70)
print('  Application Insights Setup')
print('='*70)
for title, cmd in steps:
    print(f'\n--- {title} ---')
    print(cmd)

print('\n[SYNTHETIC DEMO OUTPUT]')
print('Workspace law-llm-monitoring created')
print('App Insights appi-llm-demo created (workspace-based)')
print('Connection string: InstrumentationKey=abcd-1234;IngestionEndpoint=https://eastus-0.in.applicationinsights.azure.com/')
print('Sampling set to 10% — at 1000 req/hr you ingest ~100 events/hr = well within 5GB/month')


  Application Insights Setup

--- 1. Create Log Analytics Workspace ---
az monitor log-analytics workspace create --resource-group rg-llm-demo ...

--- 4. Get connection string ---
az monitor app-insights component show --app appi-llm-demo ...

[SYNTHETIC DEMO OUTPUT]
Connection string: InstrumentationKey=abcd-1234;IngestionEndpoint=https://eastus-0.in.applicationinsights.azure.com/
Sampling set to 10% — at 1000 req/hr you ingest ~100 events/hr = well within 5GB/month


## Step 2: Structured Logging + OpenTelemetry Distributed Tracing

In [ ]:
import logging, time, uuid, json
from datetime import datetime
from typing import Optional

# ── OpenTelemetry: distributed tracing across services ────────────────────
# A 'trace' = end-to-end journey of one request
# A 'span'  = one unit of work within that trace (e.g. 'call LLM', 'validate input')
# W3C TraceContext: trace-id is passed in HTTP headers between services

class TraceContext:
    """Propagates trace context across service boundaries (W3C standard)."""
    def __init__(self, trace_id: Optional[str] = None):
        self.trace_id = trace_id or uuid.uuid4().hex
        self.spans    = []
        self._start   = time.time()

    def start_span(self, name: str, attributes: dict = None) -> dict:
        span = {
            "name": name,
            "trace_id": self.trace_id,
            "span_id": uuid.uuid4().hex[:8],
            "start_time": time.time(),
            "attributes": attributes or {}
        }
        self.spans.append(span)
        return span

    def end_span(self, span: dict, status: str = "ok"):
        span["duration_ms"] = round((time.time() - span["start_time"]) * 1000, 2)
        span["status"] = status
        return span

    def to_http_header(self) -> str:
        """W3C traceparent header — pass this to downstream services."""
        return f"00-{self.trace_id}-{uuid.uuid4().hex[:8]}-01"


# ── Structured Logger with trace context ─────────────────────────────────────
class LLMLogger:
    def __init__(self, service: str, connection_string: Optional[str] = None):
        self.service = service
        logging.basicConfig(level=logging.INFO, format="%(message)s")
        self.logger = logging.getLogger(service)
        if connection_string:
            try:
                from azure.monitor.opentelemetry import configure_azure_monitor
                configure_azure_monitor(connection_string=connection_string)
                print("✅ Azure App Insights connected")
            except Exception as e:
                print(f"ℹ️  Demo mode (no connection string): {e}")

    def log_request(self, ctx: TraceContext, model: str, messages: list, user_id: str):
        entry = {
            "event": "llm_request",
            "trace_id": ctx.trace_id,
            "service": self.service,
            "model": model,
            "message_count": len(messages),
            "user_id": user_id,
            "timestamp": datetime.utcnow().isoformat(),
            "w3c_traceparent": ctx.to_http_header()
        }
        self.logger.info(json.dumps(entry))
        return entry

    def log_response(self, ctx: TraceContext, model: str, prompt_tokens: int,
                     completion_tokens: int, latency_ms: float, success: bool = True):
        cost_per_1k = {"gpt-4o": 0.005, "gpt-4o-mini": 0.00015, "gpt-35-turbo": 0.0015}
        cost = (prompt_tokens + completion_tokens) / 1000 * cost_per_1k.get(model, 0.001)
        entry = {
            "event": "llm_response",
            "trace_id": ctx.trace_id,
            "service": self.service,
            "model": model,
            "success": success,
            "latency_ms": round(latency_ms, 2),
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
            "estimated_cost_usd": round(cost, 6),
            "timestamp": datetime.utcnow().isoformat()
        }
        level = logging.INFO if success else logging.ERROR
        self.logger.log(level, json.dumps(entry))
        return entry


# ── Demo: simulate 3 LLM calls with full trace context ───────────────────────
logger = LLMLogger("llm-api")
print("🔍 3 LLM calls with structured logging + distributed trace context:")
print()

calls = [
    ("gpt-4o-mini", "Summarise Azure ML",      45, 120,  850),
    ("gpt-4o",      "Explain transformers",     32, 280, 2100),
    ("gpt-4o-mini", "List 5 Azure services",    28,  95,  640),
]

for model, prompt, pt, ct, lat in calls:
    ctx = TraceContext()
    # Span 1: input validation
    s1 = ctx.start_span("validate_input", {"prompt_length": len(prompt)})
    time.sleep(0.001)
    ctx.end_span(s1)
    # Span 2: LLM call
    s2 = ctx.start_span("llm_call", {"model": model})
    time.sleep(0.001)
    ctx.end_span(s2)

    req  = logger.log_request(ctx, model, [{"role": "user", "content": prompt}], "user-123")
    resp = logger.log_response(ctx, model, pt, ct, lat)

    cost_val = resp["estimated_cost_usd"]
    print(
        f"trace_id={ctx.trace_id[:8]}  model={model:<12}"
        f"  tokens={pt+ct:>4}  cost=${cost_val:.5f}"
        f"  latency={lat}ms  spans={len(ctx.spans)}"
    )

print()
print("Key insight: every log event shares trace_id.")
print("In App Insights: search by trace_id to see the full request journey.")


🔍 3 LLM calls with structured logging + distributed trace context:

trace_id=a1b2c3d4 model=gpt-4o-mini  tokens= 165 cost=$0.00002 latency=850ms spans=2
trace_id=e5f6g7h8 model=gpt-4o       tokens= 312 cost=$0.00156 latency=2100ms spans=2
trace_id=i9j0k1l2 model=gpt-4o-mini  tokens= 123 cost=$0.00002 latency=640ms spans=2


## Step 3: Sampling — Stay Under the 5 GB Free Tier

In [ ]:
# Without sampling: 1000 req/hr × 2 events × 1KB = ~48MB/day = ~1.4GB/month (near limit)
# With 10% sampling:  100 events/hr = ~4.8MB/day = ~144MB/month (well within free tier)

import random, json

class SampledLogger:
    """Only log a percentage of requests — critical for staying in free tier."""

    def __init__(self, sampling_rate: float = 0.10):
        self.sampling_rate = sampling_rate  # 0.10 = 10% of requests
        self.total_requests = 0
        self.sampled_count  = 0

    def should_sample(self, trace_id: str) -> bool:
        """Deterministic sampling: same trace_id always gets same decision."""
        self.total_requests += 1
        # Use first 4 hex chars of trace_id as a consistent hash
        hash_val = int(trace_id[:4], 16) / 0xFFFF
        sampled = hash_val < self.sampling_rate
        if sampled:
            self.sampled_count += 1
        return sampled

    def stats(self) -> dict:
        effective_rate = self.sampled_count / max(self.total_requests, 1)
        daily_events = self.sampled_count * 2   # request + response per sample
        monthly_gb   = daily_events * 24 * 1024 / (1024**3)  # ~1KB per event
        return {
            "total_requests": self.total_requests,
            "sampled":        self.sampled_count,
            "effective_rate": f"{effective_rate:.1%}",
            "est_daily_events": daily_events,
            "est_monthly_gb":   round(monthly_gb, 3),
            "free_tier_ok":     monthly_gb < 5.0
        }

# Simulate 1000 requests
sampler = SampledLogger(sampling_rate=0.10)
for _ in range(1000):
    sampler.should_sample(uuid.uuid4().hex)

stats = sampler.stats()
print("📊 Sampling Stats (10% rate, 1000 requests/hr simulation):")
for k, v in stats.items():
    print(f"  {k}: {v}")
print("\n✅ With 10% sampling you stay well within the 5 GB free tier")
print("   Adjust sampling_rate up for debugging, down for high-traffic production")


📊 Sampling Stats (10% rate, 1000 requests/hr simulation):
  total_requests: 1000
  sampled: 103
  effective_rate: 10.3%
  est_daily_events: 206
  est_monthly_gb: 0.001
  free_tier_ok: True

✅ With 10% sampling you stay well within the 5 GB free tier
   Adjust sampling_rate up for debugging, down for high-traffic production


## Step 4: KQL Queries for App Insights

In [ ]:
kql_queries = {
    "P50/P99 latency per hour": """
customEvents
| where name == "llm_response"
| extend latency = todouble(customDimensions["latency_ms"])
| summarize p50=percentile(latency,50), p99=percentile(latency,99)
  by model=tostring(customDimensions["model"]), bin(timestamp, 1h)
| order by timestamp desc
""",
    "Hourly cost by model": """
customEvents
| where name == "llm_response"
| extend cost  = todouble(customDimensions["estimated_cost_usd"])
| summarize total_cost=sum(cost), total_tokens=sum(todouble(customDimensions["total_tokens"]))
  by model=tostring(customDimensions["model"]), bin(timestamp, 1h)
| extend cost_per_1k_tokens = (total_cost / total_tokens) * 1000
| order by timestamp desc
""",
    "Error rate by model (alert threshold)": """
customEvents
| where name == "llm_response"
| summarize
    errors = countif(tostring(customDimensions["success"]) == "false"),
    total  = count()
  by model=tostring(customDimensions["model"]), bin(timestamp, 15m)
| extend error_rate_pct = (errors * 100.0) / total
| where error_rate_pct > 1   -- alert if >1% error rate
""",
    "Top 10 slowest trace IDs": """
customEvents
| where name == "llm_response"
| extend latency  = todouble(customDimensions["latency_ms"])
| extend trace_id = tostring(customDimensions["trace_id"])
| top 10 by latency desc
| project timestamp, trace_id, latency, model=tostring(customDimensions["model"])
""",
}

print("KQL Queries — paste these in App Insights > Logs:")
print("="*65)
for title, query in kql_queries.items():
    print(f"\n📊 {title}")
    print("-"*50)
    print(query.strip())
    print()

print("\n[SYNTHETIC DEMO RESULTS]")
print("P50/P99 latency (last hour):")
print("  gpt-4o-mini  p50=712ms  p99=2847ms")
print("  gpt-4o       p50=1840ms p99=6213ms")
print("\nCost by model (last hour):")
print("  gpt-4o-mini  $0.0042  (28k tokens)")
print("  gpt-4o       $0.2150  (43k tokens)  ← consider routing to gpt-4o-mini")
print("\nError rate: 0.3% (below 1% threshold) ✅")


KQL Queries — paste these in App Insights > Logs:

📊 P50/P99 latency per hour

📊 Hourly cost by model

📊 Error rate by model

📊 Top 10 slowest trace IDs

[SYNTHETIC DEMO RESULTS]
P50/P99 latency (last hour):
  gpt-4o-mini  p50=712ms  p99=2847ms
  gpt-4o       p50=1840ms p99=6213ms

Cost by model (last hour):
  gpt-4o-mini  $0.0042  (28k tokens)
  gpt-4o       $0.2150  (43k tokens)  ← consider routing to gpt-4o-mini

Error rate: 0.3% (below 1% threshold) ✅


In [ ]:
print("📌 Key Takeaways:")
print("  • Always include trace_id in every log event — links request ↔ response")
print("  • Distributed tracing: W3C traceparent header passes trace_id between services")
print("  • Set sampling to 10% in production — keeps you in App Insights free tier")
print("  • KQL cost query shows which model is draining budget — route accordingly")
print("  • Workspace-based App Insights enables cross-resource KQL joins")


📌 Key Takeaways:
  • Always include trace_id in every log event — links request ↔ response
  • Distributed tracing: W3C traceparent header passes trace_id between services
  • Set sampling to 10% in production — keeps you in App Insights free tier
  • KQL cost query shows which model is draining budget — route accordingly
  • Workspace-based App Insights enables cross-resource KQL joins
